# P6: O'zbek korpusida Word2Vec (CBOW) ni noldan o'qitamiz
**Mavzu:** neyron so'z embeddinglari; CBOW o'qitish; embedding sifatini baholash
**Kun:** 7-kun amaliyoti — 25-iyun 2026, 09:30-10:50 (80 daqiqa)
**Juftlashgan ma'ruza:** L6 — Word2Vec (CBOW), neyron embeddinglar (`d06_word2vec.pdf`)
**Kapstone modul:** m06 `CustomWord2Vec`
**Ma'lumot:** uz_news_full (online). Offline: `d07_checkpoints/uz_w2v_corpus.txt` (kichik korpus).

## Bugungi maqsadlar (ma'ruza [C] dan)
1. CBOW proyeksiya qadamini (kontekst o'rtachasi) qo'lda va kodda hisoblaymiz.
2. O'z korpusimizda CBOW modelini noldan o'qitamiz.
3. Embedding sifatini `most_similar` va analogiya bilan baholaymiz.
4. Modelni saqlab, qayta yuklaymiz va kapstone m06 moduliga ulaymiz.

## Vaqt taqsimoti
| Bo'lim | Vaqt | Mazmun |
|---|---|---|
| §1 Muhit | 4 daq | importlar, seedlar, OFFLINE_FALLBACK, HAS_GENSIM |
| §2 Yaxlit natija | 8 daq | tayyor model — most_similar demo |
| §3 Periferiya (PRIMM) | 17 daq | m01 tozalash + LineSentence; TensorBoard metadata |
| §4 Yadro | 35 daq | CBOW proyeksiya (qulflangan), o'qitish, most_similar, saqlash |
| §5 Loyihaga ulash | 13 daq | m06 moduli, import test, git |
| §6 Tadqiqot + yakun | 7 daq | agglutinatsiya va vocab explosion |

## 1. Muhit tekshiruvi
Seedlar, offline bayroq va m01 yo'li. CBOW uchun GPU shart emas.

In [ ]:
# Kaggle: pip install gensim==4.3.* (oldindan o'rnatilgan boladi)
import os, sys, random, math, tempfile
import numpy as np

random.seed(42)
np.random.seed(42)

OFFLINE_FALLBACK = True          # bundled kichik korpus bilan uchdan-uchgacha ishlaydi

# gensim ixtiyoriy: bor bo'lsa Kaggle yo'li, yo'q bo'lsa toza-numpy CBOW
try:
    import gensim
    HAS_GENSIM = True
    print("gensim mavjud:", gensim.__version__)
except ImportError:
    HAS_GENSIM = False
    print("gensim yo'q — toza-numpy CBOW ishlatiladi (offline rejim).")

# m01 (TextPreprocessor) — kapstone modullari yo'li
MODULES = os.path.join("..", "..", "capstone", "modules")
if not os.path.isdir(MODULES):
    MODULES = os.path.join("capstone", "modules")     # repo ildizidan
sys.path.insert(0, MODULES)

from m01_text_preprocessor import TextPreprocessor
print("numpy:", np.__version__, "| OFFLINE_FALLBACK =", OFFLINE_FALLBACK)


## 2. Yaxlit natija (avval manzil)
Quyida tayyor CBOW modeli o'qitiladi va `most_similar` natijasini ko'rsatamiz —
bir xil kontekstda uchragan so'zlar bir-biriga yaqin vektor oladi. Tafsilotlarni
keyingi bo'limlarda ochamiz.

In [ ]:
from m06_custom_word2vec import CustomWord2Vec

CORPUS_PATH = os.path.join("..", "..", "course", "practices", "d07_checkpoints", "uz_w2v_corpus.txt")
if not os.path.exists(CORPUS_PATH):
    CORPUS_PATH = os.path.join("course", "practices", "d07_checkpoints", "uz_w2v_corpus.txt")

_pre = TextPreprocessor()
_raw = [l.strip() for l in open(CORPUS_PATH, encoding="utf-8") if l.strip()]
_sents = [_pre.preprocess(l) for l in _raw]     # m01 bilan tozalash

_demo = CustomWord2Vec()
_demo.train(_sents, vector_size=50, window=2, min_count=2, epochs=30)
print("toshkent ga eng o'xshash:", [w for w, _ in _demo.most_similar("toshkent", n=3)])
print("telefon ga eng o'xshash:", [w for w, _ in _demo.most_similar("telefon", n=3)])


## 3. Tayyor kod bloki — korpusni tozalash va tayyorlash (PRIMM)
Bu bo'lim — bugungi o'rganish maqsadi EMAS (periferiya). Word2Vec kirishi
**tokenlangan jumlalar ro'yxati** (`list[list[str]]`) bo'lishi kerak. Buni
m01 (`TextPreprocessor`) bilan tayyorlaymiz — gensim'da bu `LineSentence` vazifasi.

**Bashorat qiling:** quyidagi katak xom korpusni tozalagach, birinchi jumla nechta
tokendan iborat bo'ladi? Stop-so'zlar (`va`, `bu`) qoladimi?

In [ ]:
# PERIFERIYA — to'liq beriladi. m01 bilan tozalash (LineSentence ekvivalenti).
def korpusni_yuklash(path):
    """Xom matn faylini m01 bilan tozalab, tokenlangan jumlalar ro'yxatini qaytaradi."""
    pre = TextPreprocessor()
    sentences = []
    for line in open(path, encoding="utf-8"):
        line = line.strip()
        if not line:
            continue
        toks = pre.preprocess(line)      # tokenize + stopword + stemming
        if toks:
            sentences.append(toks)
    return sentences

sentences = korpusni_yuklash(CORPUS_PATH)
print("Jumlalar soni:", len(sentences))
print("Birinchi jumla:", sentences[0])
print("Lug'at hajmi (taxminan):", len({w for s in sentences for w in s}))


**Tekshiring:**
1. `sentences[0]` da `va` so'zi bormi? Nega? (Maslahat: m01 stop-so'zlarni oladi.)
2. CBOW uchun nega tokenlangan jumlalar kerak, xom matn emas?
3. `min_count` parametri kam uchraydigan so'zlarga nima qiladi?

**O'zgartiring:** `CORPUS_PATH` ni o'zingizning korpusingizga (`.txt`, har qatorda bir
jumla) yo'naltiring va lug'at hajmi qanday o'zgarishini kuzating.

In [ ]:
# PERIFERIYA — TensorBoard projector metadata (Kaggle'da vizualizatsiya uchun).
# Offline: shunchaki metadata faylini yozamiz (TensorBoard ishga tushirish shart emas).
def tensorboard_metadata_yoz(model, path="w2v_metadata.tsv", limit=100):
    """Embedding projector uchun so'zlar ro'yxatini TSV ga yozadi."""
    words = model._words[:limit] if hasattr(model, "_words") and model._words else []
    if HAS_GENSIM and getattr(model, "_gensim_model", None) is not None:
        words = list(model._gensim_model.wv.index_to_key)[:limit]
    with open(path, "w", encoding="utf-8") as f:
        f.write("word\n")
        for w in words:
            f.write(w + "\n")
    return len(words)

_n = tensorboard_metadata_yoz(_demo)
print("TensorBoard metadata yozildi:", _n, "so'z")


### Checkpoint A
Agar orqada qolsangiz — quyidagi katak korpusni qaytadan yuklaydi (toza holat).

In [ ]:
if OFFLINE_FALLBACK or "sentences" not in dir():
    sentences = korpusni_yuklash(CORPUS_PATH)
print("Checkpoint: jumlalar tayyor —", len(sentences), "jumla")


## 4. Asosiy mavzu — CBOW ni qadam-baqadam (so'nuvchi tayanch)
**Namuna → Birgalikda → Mustaqil.** Avval ma'ruzaning qo'lda misolini takrorlaymiz.

### 4A. Namuna (men qilaman): CBOW proyeksiya qadami
Ma'ruza L6 [I2]-slaydida: kontekst = `[men, sevaman]`, markaz = `nlp`. CBOW kirishi —
kontekst vektorlarining **o'rtachasi**. Buni qo'lda hisoblaymiz va tekshiramiz.

In [ ]:
# CBOW proyeksiya: kontekst vektorlari o'rtachasi (L6 [I2])
emb_men     = np.array([0.5, 0.3])
emb_sevaman = np.array([0.1, 0.7])

cbow_input = np.mean([emb_men, emb_sevaman], axis=0)
print("CBOW kirish vektori:", cbow_input)

assert np.allclose(cbow_input, [0.3, 0.5])   # Ma'ruza L6 [I2]-slayd bilan solishtiring
print("To'g'ri! CBOW kirishi = avg([0.5,0.3],[0.1,0.7]) = [0.3, 0.5]")


### 4B. Birgalikda (biz qilamiz): CBOW modelini o'qitamiz
`CustomWord2Vec` ni o'z korpusimizda o'qitamiz. CBOW (`sg=0`) — kontekstdan markaziy
so'zni bashorat qiladi. CPU uchun kichik o'lcham va kam epoch tanlaymiz
(to'liq miqyos: `vector_size=100, epochs=10` Kaggle'da).

In [ ]:
w2v = CustomWord2Vec()
# === SIZNING KODINGIZ (taxminan 1 qator) ===
# CBOW ni sentences ustida o'qiting: vector_size=50, window=2, min_count=2, epochs=30
w2v.train(...)


In [ ]:
v = w2v.embed("toshkent")
assert len(w2v._words) > 0, "Lug'at bo'sh — train() chaqirilganini tekshiring."
assert v.shape == (50,), "Vektor o'lchami 50 bo'lishi kerak (vector_size=50)."
print("Ajoyib! Model o'qitildi — lug'at:", len(w2v._words), "so'z; vektor o'lchami:", v.shape[0])


### 4C. Birgalikda (biz qilamiz): o'xshash so'zlarni topamiz
Embedding sifatini `most_similar` bilan baholaymiz — kosinus o'xshashlik bo'yicha
eng yaqin so'zlar (L6 [G4]).

In [ ]:
# === SIZNING KODINGIZ (taxminan 1 qator) ===
# 'toshkent' ga eng o'xshash 3 so'zni most_similar bilan toping
similar = ...
print("toshkent ~", similar)


In [ ]:
assert isinstance(similar, list) and 0 < len(similar) <= 3, "Ro'yxat (<=3) qaytishi kerak."
assert all(isinstance(w, str) and -1.01 <= s <= 1.01 for w, s in similar), \
    "Har element (so'z, kosinus) bo'lishi va o'xshashlik [-1,1] da bo'lishi kerak."
print("To'g'ri! most_similar (so'z, o'xshashlik) juftlarini qaytardi.")


### 4D. Mustaqil (siz qilasiz): modelni saqlang va qayta yuklang
Modelni faylga saqlang, yangi obyektga yuklang va vektorlar mosligini hamda OOV
(lug'atdan tashqari) so'z uchun nol-vektorni tekshiring. Tayanch yo'q.

In [ ]:
# === SIZNING KODINGIZ (taxminan 6 qator) ===
# 1) w2v ni tempfile dagi .kv faylga saqlang
# 2) yangi CustomWord2Vec ga yuklang
# 3) assert: yuklangan 'toshkent' vektori aslga teng (np.allclose)
# 4) assert: OOV so'z ('zzqwerty') uchun nol-vektor (kvadratlar yig'indisi == 0)
path = os.path.join(tempfile.gettempdir(), "uz_w2v.kv")
...


## 5. Loyihaga ulash — m06 `CustomWord2Vec`
Bugungi ish allaqachon `capstone/modules/m06_custom_word2vec.py` da
(shartnoma `capstone/contracts.py`). U gensim-ixtiyoriy: Kaggle'da gensim CBOW,
offline'da toza-numpy. Quyida import qilib, shartnomaga mosligini tekshiramiz.

In [ ]:
from m06_custom_word2vec import CustomWord2Vec

m = CustomWord2Vec()
for metod in ["train", "embed", "most_similar", "save", "load"]:
    assert callable(getattr(m, metod, None)), f"m06.{metod} mavjud emas!"
print("m06 CustomWord2Vec shartnomaga mos: train / embed / most_similar / save / load")


Kapstone repozitoriyingizga saqlang:
```bash
git add capstone/modules/m06_custom_word2vec.py
git commit -m "P6: m06 CustomWord2Vec — CBOW embeddinglar"
git push
```

## 6. Tadqiqot savoli + yakun
**Mini-tajriba (agglutinatsiya va vocabulary explosion):** quyidagi katak `kitob`,
`kitobni`, `kitobga` shakllarini alohida so'z sifatida sanaydi. CBOW ularni alohida
o'rganadi — chunki so'zni **bo'linmas** deb biladi.

In [ ]:
formlar = ["kitob", "kitobni", "kitobga", "kitoblar"]
xom = "men kitobni oldim. u kitobga qaradi. kitoblar javonda. yangi kitob keldi."
toks = TextPreprocessor()._normalize(xom).replace(".", " ").split()
from collections import Counter
print("Stemmingsiz sanoq:", Counter(t for t in toks if t in formlar))
print("Eslatma: m01 stemming bu shakllarni 'kitob' ga keltiradi — lug'at kichrayadi.")


**Savol (o'ylab ko'ring):** agglutinativ o'zbek tilida har bir qo'shimchali shakl
alohida token bo'lsa, CBOW lug'ati portlaydi va kam uchragan shakllar shovqinli
vektor oladi. Bu muammoni qanday yengish mumkin? (Maslahat: subword/FastText —
keyingi mavzular; yoki m01 stemming.)

**Chiqish chiptasi:** _Bugun eng tushunarsiz qolgan narsa nima?_